# Deeper investigation — Persist streaming output

**Worked solution** · [All exercises](../../index.html) · [Setup](../../README.md)

Optional. Complete [Exercise 7](../07-checkpoint.ipynb) and its **Save and finish** cell first. This investigation uses the same saved work; it does not replace your core pipeline.

Completed answers use a separate solution workspace and do not replace participant work.

Run the supplied setup first. End with **Save and finish**; the next notebook loads your saved functions, so this kernel can be closed.

## Setup — supplied

Select the lab's `.venv` kernel. Close the previous exercise after **Save and finish**. Missing earlier work? Use an explicit [catch-up step](../../RECOVERY.md).

In [1]:
from pathlib import Path
import sys

# Support opening the complete repository or its labs folder in VS Code.
LAB_ROOT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / 'labs')
     if (candidate / 'workshop_runtime.py').is_file()),
    None,
)
if LAB_ROOT is None:
    raise FileNotFoundError('Open the complete labs project in VS Code; a notebook alone is not enough.')
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

from uuid import uuid4

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

import lab_checks as check
from arrival_files import publish_arrival
from lab_checks import todo
from lab_workspace import Workspace
from workshop_runtime import DATA_ROOT, create_spark, finish_query, new_run, spark_path

workspace = Workspace(solutions=True)
product_key, clean_products, clean_sales, accepted_sales, rejected_sales, enrich_sales, category_totals = workspace.load('product_key', 'clean_products', 'clean_sales', 'accepted_sales', 'rejected_sales', 'enrich_sales', 'category_totals')
RUN_ROOT, TABLE_NAME = workspace.resume_stream(after=3)
spark = create_spark(new_run("session"))
INCOMING = RUN_ROOT / "incoming"
raw = spark.read.parquet(spark_path(DATA_ROOT / "sales.parquet"))
raw_products = spark.read.parquet(spark_path(DATA_ROOT / "products.parquet"))
products = clean_products(raw_products)
cleaned = clean_sales(raw)
accepted = accepted_sales(cleaned)
rejected = rejected_sales(cleaned)
enriched = enrich_sales(accepted, products)
report = category_totals(enriched)
stream_raw = spark.readStream.schema(raw.schema).parquet(spark_path(INCOMING))
stream_enriched = enrich_sales(accepted_sales(clean_sales(stream_raw)), products)
stream_report = category_totals(stream_enriched)
CHECKPOINT_PATH = spark_path(RUN_ROOT / "report-checkpoint")
print(f"Spark {spark.version}; inputs: {DATA_ROOT.name}; notebook ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 17:35:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0; inputs: data; notebook ready


---
<a id="extension-output"></a>
## Your task

Use the existing `stream_enriched` (accepted sale rows, before aggregation) to start a separate Append-mode Parquet query as `file_query`. Supply the provided new checkpoint and output path, use `availableNow=True`, and start it.

This query processes the three files already present, then stops. Read its output as `stored_rows`. Verify five distinct sale IDs and a report equal to the batch result. The aggregate memory sink remains separate.

In [2]:
ROWS_CHECKPOINT = spark_path(RUN_ROOT / "rows-checkpoint")
ROWS_OUTPUT = spark_path(RUN_ROOT / "streamed-sales")

In [3]:
file_query = (
    stream_enriched.writeStream.format("parquet")
    .outputMode("append")
    .option("checkpointLocation", ROWS_CHECKPOINT)
    .trigger(availableNow=True)
    .start(ROWS_OUTPUT)
)

In [4]:
finish_query(file_query)

In [5]:
stored_rows = spark.read.parquet(ROWS_OUTPUT)

In [6]:
check.stored_rows(stored_rows)
check.same_report(report, category_totals(stored_rows))

Stored streaming rows verified: five distinct accepted sales.


Reports agree: five sales, total 100.00.


<details><summary>Hint</summary>

Start from `.writeStream` on the accepted enriched rows. Append-mode files contain new rows, not successive whole-report snapshots. Use the supplied bounded waiting helper.

</details>

<a id="finish"></a>
## Save and finish

Run once the core checks pass, whether or not you did the optional section. This saves your functions or stream handoff, then stops this notebook’s queries and Spark. Your work remains in `learner_work/`.

In [7]:
for active_query in spark.streams.active:
    active_query.stop()
spark.stop()
print("Session stopped; exercise files are under", RUN_ROOT.relative_to(LAB_ROOT))

Session stopped; exercise files are under runs/run-ec55513c27


Return to [all exercises](../../index.html).